# GPT Action 库 - 通过 Adzviser 接入 Google Ads

## 简介

本页面为开发者针对特定应用构建 GPT Action 提供说明和指南。在继续之前，请先熟悉以下内容： 
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零开始构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

本指南介绍如何将 **Google Ads** 报告数据连接到 ChatGPT，以获取广告系列、广告组或广告层级的关键效果指标，例如展示次数、点击次数和费用。为简化这一过程，你将使用 [Adzviser](https://adzviser.com) 作为中间件，它会确保从 Google Ads API 返回的数据格式正确，可直接在 ChatGPT 的[数据分析](https://help.openai.com/en/articles/8437071-data-analysis-with-chatgpt)环境中进行分析。

**Adzviser 的工作原理：** 首先，通过 [OAuth](https://docs.adzviser.com/getStarted/workspace) 将你的 Google Ads 账户连接到 [Adzviser](https://adzviser.com/set-up)。当你在 ChatGPT 中提出“上个月每个广告系列我花了多少钱？”这类问题时，Adzviser 会发送一个 [Google Ads 查询语言](https://developers.google.com/google-ads/api/docs/query/overview)请求，并将响应转换为 CSV 文件（小于 10MB）。然后该文件会被[返回给 ChatGPT](https://platform.openai.com/docs/actions/sending-files/returning-files) 进行分析。借助 Adzviser，你可以轻松查看和分析广告系列效果，同时基于历史数据洞察集思广益，制定优化策略。


### 价值 + 业务用例示例

**价值**：Google Ads 营销人员现在可以利用 ChatGPT 的自然语言能力，轻松查询效果指标和账户设置，而无需浏览 Google Ads 界面。整个过程中无需上传或下载任何文件。

**用例示例**： 
- 一位电商企业主希望快速查看其 Google Ads 广告系列上个月的广告支出回报率（ROAS）
- 一位品牌营销人员希望利用过去 3 个月的报告数据进行关键词和搜索字词分析，以确定哪些关键词应暂停或加大投放，以及哪些搜索字词应添加为否定关键词。
- 一位代理商营销人员需要生成一份月度报告，包含每次点击费用（CPC）、每次转化费用（CPA）和搜索展示份额等关键指标，并进行环比对比。
- 一位自由职业营销人员需要在新客户入驻过程中审核其 Google Ads 账户，以评估效果并找到优化机会。

## 演示/示例

![GPT 搜索字词分析第 1 部分](../../../images/gptactions_googleads_search_term_analysis_1.png)![GPT 搜索字词分析第 2 部分](../../../images/gptactions_googleads_search_term_analysis_2.png)

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- 如何在 Adzviser 上创建工作区：https://docs.adzviser.com/getStarted/workspace
- Adzviser 自定义 GPT 文档：https://docs.adzviser.com/chatgpt/expert
- Google Ads 提示词库：https://docs.adzviser.com/chatgpt/googleAdsPromptTemplates

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 确认你拥有某个 Google Ads 账户的[只读、标准或管理员访问权限](https://support.google.com/google-ads/answer/9978556?hl=en)。
- 在 Adzviser 上[注册](https://adzviser.com/signup)账户并激活订阅（起价 0.99 美元）。
- 通过创建工作区，将你的 Google Ads 账户连接到 Adzviser

## ChatGPT 步骤

### 自定义 GPT 指令 

创建自定义 GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，了解此步骤的详细工作方式。

In [ ]:
***Context***:
You are a Google Ads specialist who audits account health, retrieves real-time reporting data, and optimizes performances for marketers. When asked for an audit on account health, collect the relevant account settings, provide recommendations to adjust account structures. When asked about reporting data insights, gather relevant metrics and breakdowns, thoroughly analyze the reporting data, and then provide tailored recommendations to optimize performance.

***Instructions for Retrieval of Reporting Data***:
- Workflow to fetch real-time reporting data
Step 1. Calculate the date range with Python and Code Interpreter based on user input, such as "last week", "last month", "yesterday", "last 28 days", "last quarter" or "last year" etc. If no specific timeframe is provided, ask the user to clarify. Adjust for calendar variations. For example, "last week" should cover Monday to Sunday of the previous week. 
Step 2. Retrieve workspace information using the 'getWorkspace' function. 
Step 3. Fetch the relevant metrics and breakdowns for the inquired data source using functions like 'getGoogleAdsMetricsList' and 'getGoogleAdsBreakdownsList'.
Step 4. Use 'searchQuery' function with the data gathered from the previous steps like available workspace_name and metrics/breakdowns as well as calculated date range to retrieve real-time reporting data.
- Time Granularity: If the user asks for daily/weekly/quarterly/monthly data, please reflect such info in the field time_granularity in searchQueryRequest. No need to add time_granularity if the user did not ask for it explicitly.
- Returned Files: If multiple files are returned, make sure to read all of them. Each file contains data from a segment in a data source or a data source. 
- Necessary Breakdowns Only: Add important breakdowns only. Less is more. For example, if the user asks for "which ad is performing the best in Google Ads?", then you only add "Ad Name" in the breakdown list for the google_ads_request. No need to add breakdowns such as "Device" or "Campaign Name". 

***Instruction for Auditing****:
- Workflow to audit Google Ads account
Step 1. Retrieve workspace information using the 'getWorkspace' function.
Step 2. Use '/google_ads_audit/<specfic_section_to_check>' function to retrieve account settings.
- Comprehensive Audit: When asked for an comprehensive audit, don't call all the /google_ads_audit/<specfic_section_to_check> all at once. Show the users what you're planning to do next first. Then audit two sections from the Google Ads Audit GPT Knowledge at a time, then proceed to the next two sections following users consent. For the line items in the tables in the Audit Knowledge doc that don't have automation enabled, it is very normal and expected that no relevant data is seen in the retrieved response. Please highlight what needs to be checked by the user manually because these non-automated steps are important too. For example, when checking connections, adzviser only checks if the google ads account is connected with Google Merchant Center. For other connections such as YT channels, please politely ask the user to check them manually. 

***Additional Notes***:
- Always calculate the date range please with Code Interpreter and Python. It often is the case that you get the date range 1 year before when the user asks for last week, last month, etc. 
- If there is an ApiSyntaxError: Could not parse API call kwargs as JSON, please politely tell the user that this is due to the recent update in OpenAI models and it can be solved by starting a new conversation on ChatGPT.
- If the users asks for Google Ads data, for example, and there is only one workspace that has connected to Google Ads, then use this workspace name in the searchQueryRequest or googleAdsAuditRequest.
- During auditing, part of the process is to retrieve the performance metrics at account, campaign, ad group, keyword, and product levels, remember to also run Python to calculate the date range for last month and the previous period. For retrieving performance metrics at these 5 levels, please send 5 distinct requests with different breakdowns list for each level. More can be found in the audit knowledge doc.

### OpenAPI Schema 

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，了解此步骤的详细工作方式。

In [ ]:
{
  "openapi": "3.1.0",
  "info": {
    "title": "Adzviser Actions for GPT",
    "description": "Equip GPTs with the ability to retrieve real-time reporting data and account settings from Google Ads",
    "version": "v0.0.1"
  },
  "servers": [
    {
      "url": "https://copter.adzviser.com"
    }
  ],
  "paths": {
    "/google_ads/get_metrics_list": {
      "get": {
        "description": "Get the list of seletable Google Ads metrics, such as Cost, Roas, Impressions, etc.",
        "operationId": "getGoogleAdsMetricsList",
        "parameters": [],
        "deprecated": false,
        "security": [],
        "x-openai-isConsequential": false
      }
    },
    "/google_ads/get_breakdowns_list": {
      "get": {
        "description": "Get the list of seletable Google Ads breakdowns such as Device, Keyword Text, Campaign Name etc.",
        "operationId": "getGoogleAdsBreakdownsList",
        "parameters": [],
        "deprecated": false,
        "security": [],
        "x-openai-isConsequential": false
      }
    },
    "/search_bar": {
      "post": {
        "description": "Retrieve real-time reporting data such as impressions, cpc, etc. from marketing channels such as Google Ads, Fb Ads, Fb Insights, Bing Ads, etc.",
        "operationId": "searchQuery",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/searchQueryRequest"
              }
            }
          },
          "required": true
        },
        "deprecated": false,
        "security": [
          {
            "oauth2": []
          }
        ],
        "x-openai-isConsequential": false
      }
    },
    "/workspace/get": {
      "get": {
        "description": "Retrieve a list of workspaces that have been created by the user and their data sources, such as Google Ads, Facebook Ads accounts connected with each.",
        "operationId": "getWorkspace",
        "parameters": [],
        "deprecated": false,
        "security": [
          {
            "oauth2": []
          }
        ],
        "responses": {
          "200": {
            "description": "OK",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/getWorkspaceResponse"
                }
              }
            }
          }
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_merchant_center_connection": {
      "post": {
        "description": "Retrieve whether the Google Merchant Center is connected to the Google Ads account.",
        "operationId": "checkGoogleAdsMerchantCenterConnection",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_account_settings": {
      "post": {
        "description": "Retrieve the Google Ads account settings such as whether auto tagging is enabled, inventory type, etc.",
        "operationId": "checkGoogleAdsAccountSettings",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_negative_keywords_and_placements": {
      "post": {
        "description": "Retrieve the negative keywords and placements set in the Google Ads account.",
        "operationId": "checkGoogleAdsNegativeKeywordsAndPlacements",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_remarketing_list": {
      "post": {
        "description": "Retrieve the remarketing list set in the Google Ads account.",
        "operationId": "checkGoogleAdsRemarketingList",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_conversion_tracking": {
      "post": {
        "description": "Retrieve the conversion tracking status in the Google Ads account.",
        "operationId": "checkGoogleAdsConversionTracking",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_bidding_strategy": {
      "post": {
        "description": "Retrieve the bidding strategy set for each active campaigns in the Google Ads account.",
        "operationId": "checkGoogleAdsBiddingStrategy",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_search_campaign_basic": {
      "post": {
        "description": "Retrieve the basic information of the search campaigns such as campaign structure, language targeting, country targeting, etc.",
        "operationId": "checkSearchCampaignBasic",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_search_campaign_detailed": {
      "post": {
        "description": "Retrieve the detailed information of the search campaigns such as best performing keywords, ad copies, ad extentions, pinned descriptions/headlines etc.",
        "operationId": "checkSearchCampaignDetailed",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_dynamic_search_ads": {
      "post": {
        "description": "Retrieve the dynamic search ads information such as dynamic ad targets, negative ad targets, best performing search terms etc.",
        "operationId": "checkDynamicSearchAds",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    },
    "/google_ads_audit/check_pmax_campaign": {
      "post": {
        "description": "Retrieve the performance of the pmax campaigns such as search themes, country/language targeting, final url expansions, excluded urls.",
        "operationId": "checkPmaxCampaign",
        "parameters": [],
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "$ref": "#/components/schemas/googleAdsAuditRequest"
              }
            }
          },
          "required": true
        },
        "x-openai-isConsequential": false
      }
    }
  },
  "components": {
    "schemas": {
      "getWorkspaceResponse": {
        "title": "getWorkspaceResponse",
        "type": "array",
        "description": "The list of workspaces created by the user on adzviser.com/main. A workspace can include multiple data sources",
        "items": {
          "type": "object",
          "properties": {
            "name": {
              "title": "name",
              "type": "string",
              "description": "The name of a workspace"
            },
            "data_connections_accounts": {
              "title": "data_connections_accounts",
              "type": "array",
              "description": "The list of data sources that the workspace is connected. The name can be an account name and type can be Google Ads/Facebook Ads/Bing Ads",
              "items": {
                "type": "object",
                "properties": {
                  "name": {
                    "title": "name",
                    "type": "string",
                    "description": "The name of a data connection account"
                  }
                }
              }
            }
          }
        }
      },
      "googleAdsAuditRequest": {
        "description": "Contains details about the Google Ads account audit request.",
        "type": "object",
        "required": [
          "workspace_name"
        ],
        "title": "googleAdsAuditRequest",
        "properties": {
          "workspace_name": {
            "type": "string",
            "title": "workspace_name",
            "description": "Call API getWorkspace first to get a list of available workspaces"
          }
        }
      },
      "searchQueryRequest": {
        "description": "Contains details about queried data source, metrics, breakdowns, time ranges and time granularity, etc.",
        "type": "object",
        "required": [
          "assorted_requests",
          "workspace_name",
          "date_ranges"
        ],
        "title": "searchQueryRequest",
        "properties": {
          "assorted_requests": {
            "type": "object",
            "title": "assorted_requests",
            "description": "For example, if the user asks for \"cost on Google ads last month\", then call getGoogleAdsMetricsList and getGoogleAdsBreakdownsList to retrieve the latest up-to-date info about how to compose a google_ads_request. A metric is a quantitative measurement. It represents data that can be measured and expressed in numbers. Metrics are used to track performance or behavior. Examples include clicks, impressions, conversions, revenue, etc. A breakdown is a qualitative attribute or descriptor. It provides context for metrics by categorizing or segmenting them. Breakdowns are text. Examples include country, channel, campaign name, etc. DO NOT include Date, Month, Quarter or Year in the list of breakdowns in any of the requests below. The breakdowns should be NOT mixed up with metrics, meaning that the selected breakdowns should be passed into the property \"breakdowns\", not \"metrics\", and vice versa.",
            "properties": {
              "google_ads_request": {
                "type": "object",
                "description": "DO NOT come up with metrics and breakdowns on your own. You MUST call API getGoogleAdsMetricsList and getGoogleAdsBreakdownsList to be better informed prior of composing a googleAdsRequest.",
                "required": [
                  "metrics",
                  "breakdowns"
                ],
                "properties": {
                  "breakdowns": {
                    "type": "array",
                    "items": {
                      "type": "string"
                    },
                    "description": "Must call API getGoogleAdsBreakdownsList to retrieve a list of selectable breakdowns."
                  },
                  "metrics": {
                    "type": "array",
                    "items": {
                      "type": "string"
                    },
                    "description": "Must call API getGoogleAdsMetricsList to retrieve a list of selectable metrics."
                  }
                }
              }
            }
          },
          "workspace_name": {
            "type": "string",
            "title": "workspace_name",
            "description": "Call API getWorkspace first to get a list of available workspaces. Multiple data sources (such as Google ads, Bing ads) can be stored in one workspace. If the user does not specify a workspace name, then use the available workspace name from the retrieved list and see which one has Google Ads. If the user has not yet created one, then ask them to go to adzviser.com/main to create a new workspace."
          },
          "date_ranges": {
            "type": "array",
            "description": "A list of date ranges requested from the user. They needs to be calculated seperately with Code Interpreter and Python every single time for accuracy. For example, if the user requests \"Google Ads search impression share in May and August\", then this array should be [[\"2024-05-01\", \"2024-05-31\"], [\"2024-08-01\", \"2024-08-31\"]].",
            "items": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "A 2-element array. The first represents the start date and the second the end date. Both are in YYYY-MM-DD format."
            }
          },
          "time_granularity": {
            "type": "string",
            "title": "time_granularity",
            "default": "",
            "description": "Describes how granularity you wish the date_ranges to be. For example, If the user asks \"weekly cost on Google Ads\" this year, then this value should be \"Week\". If the user does not specify, then leave it as empty.",
            "enum": [
              "Date",
              "Week",
              "Month",
              "Quarter"
            ]
          }
        }
      }
    },
    "securitySchemes": {
      "oauth2": {
        "type": "oauth2"
      }
    }
  }
}

## 认证说明

以下是与该第三方应用设置认证的说明。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，了解此步骤的详细工作方式。

![GPT OAuth 设置](../../../images/gptactions_adzviser_oauth.png)

### 在 ChatGPT 中

在 ChatGPT 中，点击 "Authentication" 并选择 **"OAuth"**。输入以下信息。 

- **Client ID**：（留空）
- **Client Secret**：（留空）
- **Authorization URL**：https://adzviser.com/authorize-gpt
- **Token URL**：https://adzviser.com/api/oauth-exchange-token-gpt
- **Scope**：（留空）
- **Token Exchange Method**：Default (POST)

### 常见问题与故障排除

- *Google Ads 账户列表为空*：如果你在尝试连接 Google Ads 账户时遇到账户列表为空的情况，很可能是你还没有为 Google Ads 账户命名。要解决此问题，请前往 ads.google.com 并登录，然后按照[这里](https://support.google.com/google-ads/answer/7519527?hl=en)的说明为你的 Google Ads 账户命名。

*有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 github 中提交 PR 或 issue，我们会进行查看。*